# 3.10 · 数据划分与交叉验证 / Splitting & Cross-Validation

> **课程定位 / Where this fits**
> 第 10 课，**Part 3 · EDA 与数据预处理**。
> Lesson 10, **Part 3 · EDA & Preprocessing**.
>
> 3.9 说泄漏的解药是"正确的划分"，这一课讲透怎么划分、怎么评估。**交叉验证(CV)** 是评估模型最重要的工具——它决定了你对模型能力的估计准不准，也是模型选择和调参的基础。
> 3.9 said the antidote to leakage is "correct splitting"; this lesson covers it fully. **Cross-validation (CV)** is the single most important tool for evaluating a model — it determines whether your estimate of model quality is trustworthy, and underpins model selection and tuning.
>
> 💼 **实战/面试视角**："为什么用交叉验证 / 分层和分组 CV 的区别 / 嵌套 CV 解决什么" 是必考。
> 💼 **Practical/interview angle:** "why cross-validate / stratified vs group CV / what nested CV solves" are must-knows.

> 💡 **面试相关 / Interview-relevant**
> - "为什么用 K-fold 而不是单次划分"（出镜率 ★★★★★）
> - "StratifiedKFold 解决什么 / 何时用"（★★★★★）
> - "GroupKFold / TimeSeriesSplit 何时用"（★★★★★）
> - "嵌套交叉验证解决什么（调参乐观偏差）"（★★★★★）
> - "train/val/test 三分的作用"（★★★★）

---

## 学习目标 / Learning Objectives

1. 理解单次划分的"运气"问题，**为什么要 K-fold**。
   Understand single-split luck and **why we K-fold**.
2. 掌握 **StratifiedKFold**（保持类别比例）。
   Master StratifiedKFold (preserves class ratio).
3. 掌握 **GroupKFold**（重复实体）与 **TimeSeriesSplit**（时序）。
   Master GroupKFold (repeated entities) and TimeSeriesSplit (time series).
4. 理解 train/val/test 三分，以及调参在 val 上的**乐观偏差**。
   Understand train/val/test and the **optimism** of tuning on val.
5. 用**嵌套 CV** 得到调参后的无偏估计。
   Use **nested CV** for an unbiased estimate after tuning.

## 目录 / TOC
1. [先建直觉：为什么 K-fold ⭐](#1)
2. [KFold 与 shuffle 陷阱 ⭐](#2)
3. [StratifiedKFold：不平衡必备 ⭐](#3)
4. [GroupKFold：重复实体 ⭐](#4)
5. [TimeSeriesSplit：时序 ⭐](#5)
6. [调参的乐观偏差 ⭐](#6)
7. [嵌套 CV ⭐](#7)
8. [小结](#8)


<a id="1"></a>
## 1. 先建直觉：为什么 K-fold ⭐ / Intuition: Why K-fold

只做**一次** train/test 划分，分数会受"**这次恰好分到哪些样本当测试**"的运气影响。简单的样本进了测试集，分数就虚高；难的进了，就虚低。下面用 20 个不同随机种子做单次划分，看分数能波动多大。
A **single** train/test split makes the score depend on the luck of "which samples happened to land in the test set". Easy samples in test → inflated; hard ones → deflated. Below we use 20 random seeds to see how much a single split's score swings.

**K 折交叉验证(K-fold CV)** 的解法：把数据切成 K 份，轮流用其中 1 份当验证、其余 K−1 份训练，做 K 次，**取平均**。这样每个样本都当过一次验证，分数更稳、还附带一个标准差告诉你波动有多大。
**K-fold CV** fixes this: split the data into K parts, take turns using 1 as validation and the other K−1 for training, K times, and **average**. Every sample serves as validation once; the score is more stable and comes with a std telling you the spread.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import (train_test_split, cross_val_score, KFold,
                                     StratifiedKFold, GroupKFold, TimeSeriesSplit)
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

X, y = load_iris(return_X_y=True)

# 单次划分的"运气": 20 个不同 random_state, 同模型同数据 / single-split luck across 20 seeds
scores = []
for seed in range(20):
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=seed)
    scores.append(LogisticRegression(max_iter=500).fit(Xtr, ytr).score(Xte, yte))
print("20 个不同随机种子的单次 test 准确率:")
print(f"  范围 [{min(scores):.1%}, {max(scores):.1%}], std={np.std(scores):.1%}")
print(f"  → 同模型同数据, 仅换 test 划分, 波动 {(max(scores)-min(scores))*100:.0f} 个百分点!")

# 5 折 CV 一次给出更稳的均值±std / K-fold gives a stable mean±std
cv = cross_val_score(LogisticRegression(max_iter=500), X, y, cv=5)
print(f"\n5-fold CV: {cv.mean():.1%} ± {cv.std():.1%}  (每个样本都当过一次验证, 更可靠)")


<a id="2"></a>
## 2. KFold 与 shuffle 陷阱 ⭐ / KFold & the Shuffle Trap

`KFold` 默认**按顺序**切分。如果数据**本身是排好序的**（如 Iris 按类别排列），不打乱就会让某些折**全是同一个类**，评估完全失真。所以：**用 KFold 前要么 `shuffle=True`，要么对分类任务直接用分层（下一节）。**
`KFold` splits **in order** by default. If the data is **already sorted** (Iris is sorted by class), not shuffling makes some folds **entirely one class**, ruining evaluation. So: **with KFold, either `shuffle=True`, or use stratification for classification (next section).**


In [ ]:
# 可视化 K 折结构: 每行一折, 红色=该折的验证样本 / visualize fold structure
def plot_cv(cv, X, y, groups=None, ax=None, title=""):
    n = len(X)
    for i, (tr, te) in enumerate(cv.split(X, y, groups)):
        idx = np.zeros(n); idx[te] = 1                  # 1=验证集(红), 0=训练集(蓝)
        ax.scatter(range(n), [i]*n, c=idx, cmap="coolwarm", marker="_", lw=8, vmin=0, vmax=1)
    ax.set_title(title); ax.set_xlabel("样本索引 sample index"); ax.set_ylabel("fold")
    ax.set_yticks(range(cv.get_n_splits(X, y, groups)))

fig, axes = plt.subplots(1, 2, figsize=(13, 3))
plot_cv(KFold(5), X, y, ax=axes[0], title="KFold (按顺序切, 红=验证)")
plot_cv(KFold(5, shuffle=True, random_state=0), X, y, ax=axes[1], title="KFold(shuffle=True) 打乱后")
plt.tight_layout(); plt.show()
print("⚠ Iris 按类别排序! 不 shuffle 的 KFold 会让某些折全是一个类 → 必须 shuffle 或用分层")


<a id="3"></a>
## 3. StratifiedKFold：不平衡必备 ⭐ / StratifiedKFold

**分层(stratified)** 划分保证每一折里**各类别的比例**和整体一致。对**不平衡**数据这是必须的——否则少数类可能在某些折里一个都没有，根本无法评估。**分类任务默认就该用 `StratifiedKFold`**（sklearn 的 `cross_val_score` 对分类器自动用它）。
**Stratified** splitting keeps each class's **proportion** the same in every fold. For **imbalanced** data this is mandatory — otherwise the minority class may be entirely absent from some folds, making evaluation impossible. **Classification should default to `StratifiedKFold`** (sklearn's `cross_val_score` uses it automatically for classifiers).


In [ ]:
# 90%/10% 不平衡数据 / imbalanced 90/10 data
y_imb = np.array([0]*180 + [1]*20); X_imb = rng.normal(size=(200, 4))

print("普通 KFold 各折 test 里的少数类(1)数量:")
for i, (tr, te) in enumerate(KFold(5).split(X_imb)):
    print(f"  fold {i}: {(y_imb[te]==1).sum()} 个", end="  ")
print("\n  ← 某些折可能 0 个少数类, 无法评估!\n")

print("StratifiedKFold 各折 test 里的少数类(1)数量:")
for i, (tr, te) in enumerate(StratifiedKFold(5).split(X_imb, y_imb)):
    print(f"  fold {i}: {(y_imb[te]==1).sum()} 个", end="  ")
print("\n  ← 每折都有 4 个(=20/5), 比例严格保持 ✓")


<a id="4"></a>
## 4. GroupKFold：重复实体 ⭐ / GroupKFold

接 3.9 的分组泄漏：当有**重复实体**（同一用户/病人/设备多条记录）时，必须保证**同一实体的所有记录只出现在一边**。`GroupKFold` 按你给的 `groups` 来切，绝不让一个 group 跨越 train 和 test。
Continuing 3.9's group leakage: with **repeated entities** (a user/patient/device with multiple records), you must keep **all records of an entity on one side**. `GroupKFold` splits by the `groups` you provide, never letting a group straddle train and test.


In [ ]:
groups = np.repeat(np.arange(40), 5)        # 40 个病人, 每人 5 条记录 / 40 patients × 5 records
X_g = rng.normal(size=(200, 4)); y_g = rng.integers(0, 2, 200)

fig, axes = plt.subplots(1, 2, figsize=(13, 3))
plot_cv(KFold(5), X_g, y_g, ax=axes[0], title="KFold: 同病人散在多折 (泄漏!)")
plot_cv(GroupKFold(5), X_g, y_g, groups=groups, ax=axes[1], title="GroupKFold: 同病人锁定一折 ✓")
plt.tight_layout(); plt.show()

# 验证: GroupKFold 保证 train 和 test 没有共享的 group / no shared group between train & test
for tr, te in GroupKFold(5).split(X_g, y_g, groups):
    assert len(set(groups[tr]) & set(groups[te])) == 0
print("GroupKFold 保证: 每折 test 的病人在 train 里完全不出现 ✓ (杜绝'靠认人作弊')")


<a id="5"></a>
## 5. TimeSeriesSplit：时序 ⭐ / TimeSeriesSplit

接 3.9 的时序泄漏：时序数据要保证**训练集永远在验证集之前**（用过去预测未来）。`TimeSeriesSplit` 做的就是这件事：随着折数推进，训练集**不断向前扩张**，验证集永远是紧接其后的一段——完美模拟真实部署。
Continuing 3.9's temporal leakage: time series require **training to always precede validation** (past predicts future). `TimeSeriesSplit` does exactly that: as folds advance, the training set **grows forward** and validation is always the next chunk — mirroring real deployment.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 3))
X_ts = np.arange(100).reshape(-1, 1); y_ts = np.arange(100)
plot_cv(KFold(5), X_ts, y_ts, ax=axes[0], title="❌ KFold: 验证散布, 用'未来'训练'过去'")
plot_cv(TimeSeriesSplit(5), X_ts, y_ts, ax=axes[1], title="✅ TimeSeriesSplit: train 永在 val 之前")
plt.tight_layout(); plt.show()
print("TimeSeriesSplit 模拟真实部署: 用历史训练, 预测未来; 训练集随时间增长 (Part 14 时序详述)")


<a id="6"></a>
## 6. 调参的乐观偏差 ⭐ / The Optimism of Tuning

为什么需要 train/**val**/test **三**份？因为一旦你用验证集去**挑超参**（试很多 C 选最好的），这个"最好"的验证分数就**偏乐观**了——你是专门挑了在这份验证集上运气最好的那个。所以必须留一份**全程不参与挑选**的 test，才能得到诚实的估计。
Why three sets — train/**val**/test? Because once you use validation to **pick hyperparameters** (try many C, keep the best), that "best" validation score is **optimistic** — you specifically chose whatever got luckiest on that validation set. So you need a test set that **never participates in selection** for an honest estimate.


In [ ]:
from sklearn.svm import SVC
# 三分: train / val / test / split into train, val, test
X_trval, X_test, y_trval, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
X_tr, X_val, y_tr, y_val = train_test_split(X_trval, y_trval, test_size=0.25, random_state=0)

# 在 val 上试很多 C, 选 val 表现最好的 / try many C, pick best on val
best_val, best_C = 0, None
for C in np.logspace(-2, 3, 40):
    acc = SVC(C=C).fit(X_tr, y_tr).score(X_val, y_val)
    if acc > best_val: best_val, best_C = acc, C

# 用选定的 C 在 train+val 上重训, 在从未参与挑选的 test 上评估 / honest estimate on test
test_acc = SVC(C=best_C).fit(X_trval, y_trval).score(X_test, y_test)
print(f"挑出的最优 C = {best_C:.2f}")
print(f"它在 val 上的准确率: {best_val:.1%}  ← 乐观(我们专门挑的它)")
print(f"它在 test 上的准确率: {test_acc:.1%}  ← 诚实估计(test 全程没参与挑选)")


<a id="7"></a>
## 7. 嵌套 CV ⭐ / Nested CV

把"挑超参"和"评估"用**同一个 CV** 来做，会有同样的乐观偏差（`GridSearchCV.best_score_` 就偏高）。**嵌套交叉验证**用两层 CV 解决：**内层 CV 调参**，**外层 CV 评估**——外层的每一折都把"调参"当成模型的一部分重新做一遍，于是评估时从没见过被选中的超参。这是报告模型**真实能力**的金标准。
Using the **same CV** for both tuning and evaluation carries the same optimism (`GridSearchCV.best_score_` is inflated). **Nested CV** fixes it with two loops: the **inner CV tunes**, the **outer CV evaluates** — each outer fold redoes the tuning as part of the model, so evaluation never saw the chosen hyperparameters. It's the gold standard for reporting a model's **true ability**.


In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.datasets import make_classification

# 用有噪声的小数据(Iris 太干净显不出偏差) / noisy small data so the bias shows
X4, y4 = make_classification(n_samples=200, n_features=20, n_informative=5,
                             n_redundant=2, flip_y=0.15, random_state=0)
param_grid = {"C": np.logspace(-3, 3, 12)}

non_nested, nested = [], []
for seed in range(8):                                   # 多种子平均, 让偏差稳定显现
    inner = StratifiedKFold(5, shuffle=True, random_state=seed)
    outer = StratifiedKFold(5, shuffle=True, random_state=seed)
    grid = GridSearchCV(SVC(), param_grid, cv=inner).fit(X4, y4)
    non_nested.append(grid.best_score_)                 # 用同一CV既调参又报分 → 乐观
    nested.append(cross_val_score(grid, X4, y4, cv=outer).mean())  # 外层评估"含调参的整个流程"
print(f"非嵌套 CV (调参+报分同一CV): {np.mean(non_nested):.1%}  ← 乐观偏差")
print(f"嵌套 CV (内层调参, 外层评估): {np.mean(nested):.1%}  ← 无偏估计")
print(f"差异 {(np.mean(non_nested)-np.mean(nested))*100:.1f} 个百分点 = 调参偷看带来的乐观偏差")
print("→ 汇报模型真实能力用 nested CV; 选完超参再用全部 train 重训上线")


<a id="8"></a>
## 8. 小结 / Summary

```
单次划分有运气波动 → K-fold CV(轮流验证, 取均值±std) 更可靠
KFold 默认按顺序: 数据有序时必须 shuffle=True
StratifiedKFold: 保持各折类别比例, 不平衡/分类任务必备(cross_val_score 对分类器自动用)
GroupKFold: 同实体(用户/病人)锁定一折, 防分组泄漏
TimeSeriesSplit: train 永在 val 之前, 时序必用
调参乐观偏差: 用 val 挑超参后 val 分偏高 → 留独立 test
嵌套 CV: 内层调参+外层评估 → 调参后的无偏估计(汇报真实能力的金标准)
```

### 💡 面试速查 / Interview cheat-sheet
1. **K-fold 比单次划分稳**：每样本当过验证，给均值±std。
   K-fold beats a single split: every sample validates once, gives mean±std.
2. **分类用 StratifiedKFold**（保持类别比例，不平衡必备）。
   Classification → StratifiedKFold (keeps class ratio, essential when imbalanced).
3. **GroupKFold（重复实体）/ TimeSeriesSplit（时序）**——别用随机 K-fold。
   GroupKFold (repeated entities) / TimeSeriesSplit (time series) — not random K-fold.
4. **调参在 val 上偏乐观** → 留独立 test。
   Tuning on val is optimistic → keep a separate test set.
5. **嵌套 CV** 给调参后的无偏估计（汇报真实能力）。
   Nested CV gives an unbiased post-tuning estimate.

### 下一节 / Next
**3.11 不平衡数据**——划分讲完，回到不平衡这个反复出现的主题：重采样(SMOTE)、类权重、阈值移动，以及该看什么指标。
**3.11 Imbalanced Data** — back to the recurring imbalance theme: resampling (SMOTE), class weights, threshold shifting, and which metrics to watch.
